In [19]:
!pip install -q pydicom grad-cam albumentations nibabel

import os
import glob
import numpy as np
import pandas as pd
import pydicom
import cv2
import torch
import torch.nn as nn
import nibabel as nib  # <--- Nuova per le segmentazioni
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from pydicom.pixel_data_handlers.util import apply_voi_lut
from sklearn.model_selection import train_test_split

# Percorsi
BASE_PATH = "/kaggle/input/rsna-2022-cervical-spine-fracture-detection"
TRAIN_IMAGES_DIR = os.path.join(BASE_PATH, "train_images")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")
SEG_DIR = os.path.join(BASE_PATH, "segmentations") # <--- Cartella segmentazioni

In [20]:
class RSNAKaggleDataset(Dataset):
    def __init__(self, df, image_dir, seg_dir=None):
        self.df = df
        self.image_dir = image_dir
        self.seg_dir = seg_dir

    def __len__(self):
        return len(self.df)

    def get_anatomical_centers(self, uid, num_slices):
        """
        PARTE NUOVA: Estrae i centri reali delle vertebre dalle segmentazioni.
        Se non trova il file, usa il vecchio metodo matematico come fallback.
        """
        seg_path = os.path.join(self.seg_dir, f"{uid}.nii.gz")
        
        # Se la segmentazione esiste, calcoliamo i centri reali
        if self.seg_dir and os.path.exists(seg_path):
            try:
                seg_img = nib.load(seg_path)
                # I file NIfTI di Kaggle sono spesso orientati diversamente, 
                # usiamo get_fdata() e cerchiamo i centri sull'asse Z (fette)
                data = seg_img.get_fdata()
                centers = []
                for v_idx in range(1, 8): # C1-C7
                    z_coords = np.where(data == v_idx)[2]
                    if len(z_coords) > 0:
                        centers.append(int(np.median(z_coords)))
                    else:
                        # Se manca una vertebra specifica nella maschera, fallback
                        centers.append(int(num_slices * (v_idx - 0.5) / 7))
                return centers
            except:
                pass
        
        # Fallback matematico (quello vecchio)
        return np.linspace(0, num_slices-1, 9).astype(int)[1:-1]

    def preprocess_slice(self, path):
        try:
            dicom = pydicom.dcmread(path)
            img = dicom.pixel_array
            if 'WindowCenter' in dicom:
                img = apply_voi_lut(img, dicom)
            img = img - np.min(img)
            if np.max(img) != 0: 
                img = img / np.max(img)
            
            img_8bit = (img * 255).astype(np.uint8)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            img = clahe.apply(img_8bit).astype(np.float32) / 255.0
            return cv2.resize(img, (224, 224))
        except:
            return np.zeros((224, 224), dtype=np.float32)

    def __getitem__(self, idx):
        uid = self.df.iloc[idx]['StudyInstanceUID']
        labels = self.df.iloc[idx][['C1','C2','C3','C4','C5','C6','C7']].values.astype(float)
        
        path = os.path.join(self.image_dir, uid)
        # Ordiniamo le fette numericamente per assicurarci che l'indice coincida con la segmentazione
        slices = sorted(glob.glob(os.path.join(path, "*.dcm")), 
                        key=lambda x: int(os.path.basename(x).replace('.dcm', '')))
        
        vertebrae_stacks = []
        if len(slices) >= 7:
            # USIAMO LA NUOVA LOGICA ANATOMICA
            centers = self.get_anatomical_centers(uid, len(slices))
            
            for mid in centers:
                # Ora prendiamo 5 fette invece di 3 per coprire meglio l'osso (2.5D esteso)
                # Le mediamo o le stackiamo? Qui le stackiamo prendendo -2, centro, +2
                s1 = self.preprocess_slice(slices[max(0, mid-2)])
                s2 = self.preprocess_slice(slices[mid])
                s3 = self.preprocess_slice(slices[min(len(slices)-1, mid+2)])
                vertebrae_stacks.append(np.stack([s1, s2, s3], axis=0))
        else:
            for _ in range(7): 
                vertebrae_stacks.append(np.zeros((3, 224, 224)))
            
        return torch.tensor(np.array(vertebrae_stacks), dtype=torch.float32), torch.tensor(labels, dtype=torch.float32)

# Modello resta uguale (ResNet18)
class MultiVertebraModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights='DEFAULT')
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        bs, nv, c, h, w = x.shape
        x = x.view(bs * nv, c, h, w) 
        features = self.encoder(x).view(bs * nv, -1)
        out = self.fc(features)
        return out.view(bs, nv)

In [21]:
def run_training():
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    df = pd.read_csv(TRAIN_CSV)
    
    # TRUCCO INTELLIGENTE: 
    # Cerchiamo quali pazienti hanno una segmentazione disponibile
    seg_ids = [f.replace('.nii.gz', '') for f in os.listdir(SEG_DIR)]
    df_with_seg = df[df['StudyInstanceUID'].isin(seg_ids)]
    df_others = df[~df['StudyInstanceUID'].isin(seg_ids)].head(200) # Prendi un po' di altri come backup
    
    # Uniamo i due (dando priorità a quelli con segmentazione reale)
    df_final = pd.concat([df_with_seg, df_others]).sample(frac=1).reset_index(drop=True)
    
    train_df, val_df = train_test_split(df_final, test_size=0.2, random_state=42)

    # Passiamo SEG_DIR al dataset
    train_loader = DataLoader(RSNAKaggleDataset(train_df, TRAIN_IMAGES_DIR, SEG_DIR), batch_size=2, shuffle=True)
    val_loader = DataLoader(RSNAKaggleDataset(val_df, TRAIN_IMAGES_DIR, SEG_DIR), batch_size=2)

    model = MultiVertebraModel().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    print(f"Training su {len(df_final)} pazienti ({len(df_with_seg)} con segmentazione reale)")
    
    model.train()
    for epoch in range(3): # Aumentiamo a 3 epoche visto che i dati sono più precisi
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            if i % 10 == 0:
                print(f"Epoca {epoch+1} | Batch {i} | Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), "/kaggle/working/cervical_model_v2_anatomical.pth")
    print("Training completato con guida anatomica!")

run_training()

Training su 200 pazienti (0 con segmentazione reale)
Epoca 1 | Batch 0 | Loss: 1.1909
Epoca 1 | Batch 10 | Loss: 0.8770
Epoca 1 | Batch 20 | Loss: 1.2205
Epoca 1 | Batch 30 | Loss: 0.5704
Epoca 1 | Batch 40 | Loss: 0.7167
Epoca 1 | Batch 50 | Loss: 0.5880
Epoca 1 | Batch 60 | Loss: 0.8276
Epoca 1 | Batch 70 | Loss: 2.2768
Epoca 2 | Batch 0 | Loss: 0.5107
Epoca 2 | Batch 10 | Loss: 0.5360
Epoca 2 | Batch 20 | Loss: 0.5755
Epoca 2 | Batch 30 | Loss: 0.5239
Epoca 2 | Batch 40 | Loss: 1.8149
Epoca 2 | Batch 50 | Loss: 0.4488
Epoca 2 | Batch 60 | Loss: 0.4925
Epoca 2 | Batch 70 | Loss: 1.0609
Epoca 3 | Batch 0 | Loss: 2.0438
Epoca 3 | Batch 10 | Loss: 0.3920
Epoca 3 | Batch 20 | Loss: 0.3666
Epoca 3 | Batch 30 | Loss: 0.6665
Epoca 3 | Batch 40 | Loss: 0.4167
Epoca 3 | Batch 50 | Loss: 0.2627
Epoca 3 | Batch 60 | Loss: 0.2054
Epoca 3 | Batch 70 | Loss: 0.2800
Training completato con guida anatomica!


In [22]:
# uguale alla cella di sopra ma piu robusto
from sklearn.model_selection import train_test_split

def run_training():
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Lavoro su: {DEVICE}")

    # 1. Carichiamo il CSV
    df = pd.read_csv(TRAIN_CSV)
    
    # 2. Per il primo test, prendo solo 100 pazienti (per essere sicuri che funzioni)
    # Se tutto va bene, potrò aumentare questo numero
    df_sample = df.head(100) 
    train_df, val_df = train_test_split(df_sample, test_size=0.2, random_state=42)

    # 3. Preparo i caricatori di dati
    train_loader = DataLoader(RSNAKaggleDataset(train_df, TRAIN_IMAGES_DIR), batch_size=2, shuffle=True)
    val_loader = DataLoader(RSNAKaggleDataset(val_df, TRAIN_IMAGES_DIR), batch_size=2)

    # 4. Inizializzo Modello, Loss e Optimizer
    model = MultiVertebraModel().to(DEVICE)
    # BCEWithLogitsLoss è ideale per classificare C1-C7 contemporaneamente
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0]).to(DEVICE))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    print("--- Inizio Training (Test su 100 pazienti) ---")
    model.train()
    for epoch in range(2): # Faccio 2 epoche per il test
        epoch_loss = 0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            if i % 10 == 0:
                print(f"Epoca {epoch+1} | Batch {i} | Loss Corrente: {loss.item():.4f}")

    # 5. Salvataggio finale nell'area di output di Kaggle
    torch.save(model.state_dict(), "/kaggle/working/cervical_model_v1.pth")
    print("--- Training Completato e Modello Salvato! ---")

# Lancio la funzione
run_training()

Lavoro su: cuda
--- Inizio Training (Test su 100 pazienti) ---


TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
#  IMPORT NECESSARI 
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import pydicom
import glob
import os
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 2. DEFINIZIONE WRAPPER
class GradCAMWrapper(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.encoder = full_model.encoder
        self.fc = full_model.fc

    def forward(self, x):
        features = self.encoder(x).view(x.size(0), -1)
        return self.fc(features)

# 3. FUNZIONE DI VISUALIZZAZIONE
def visualize_fracture_detection(model_path, patient_idx=10):
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Caricamento dati
    df = pd.read_csv(TRAIN_CSV)
    ds = RSNAKaggleDataset(df, TRAIN_IMAGES_DIR)
    images_stack, labels = ds[patient_idx] 
    
    # Caricamento modello
    base_model = MultiVertebraModel().to(DEVICE)
    base_model.load_state_dict(torch.load(model_path))
    base_model.eval()

    wrapped_model = GradCAMWrapper(base_model).to(DEVICE)
    target_layers = [wrapped_model.encoder[-1]]
    
    # importazione GradCAM
    cam = GradCAM(model=wrapped_model, target_layers=target_layers)
    
    v_idx = 3 
    input_tensor = images_stack[v_idx].unsqueeze(0).to(DEVICE)
    targets = [BinaryClassifierOutputTarget(0)]
    
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
    
    with torch.no_grad():
        prediction = torch.sigmoid(wrapped_model(input_tensor)).item()
    
    img_to_show = images_stack[v_idx].permute(1, 2, 0).cpu().numpy()
    img_to_show = (img_to_show - img_to_show.min()) / (img_to_show.max() - img_to_show.min() + 1e-9)
    visualization = show_cam_on_image(img_to_show, grayscale_cam, use_rgb=True)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(img_to_show[:,:,1], cmap='bone')
    plt.title(f"Paziente {patient_idx} - Originale")
    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.title(f"Detezione (Probabilità: {prediction:.4f})")
    plt.show()

#  ESECUZIONE
visualize_fracture_detection("/kaggle/working/cervical_model.pth", patient_idx=10)